# Stage 8. Backbone Experiment

Notebook ini membandingkan tiga backbone deep embedding (ResNet18-CSA baseline, DINOv2-S self-supervised ViT, SigLIP-B vision-language ViT) pada situs palm, memakai threshold valid_pixel_mask baru (high=235) berdasarkan analisis empiris notebooks/analysis_valid_pixel.ipynb. Palm memakai threshold lebih lunak dari conjunctiva dan nail (235 vs 220) karena 58% pixel ROI mentah memiliki channel_max >= 240, disebabkan warna kulit telapak tangan yang secara natural lebih cerah. Model produksi palm saat ini adalah Full Fusion (deep + handcrafted), sehingga eksperimen menguji Path B saja maupun Full Fusion untuk setiap backbone.

## Environment Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import json

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score

from configs import paths
from src.common import features, manifest as manifest_utils, train

output_dir = paths.outputs_dir("palm")
artifact_dir = paths.artifacts_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
manifest = manifest_utils.assign_kfold(manifest, n_splits=5, seed=42)

PIXEL_HIGH = 235

print("sampel", len(manifest))
print("device", "cuda" if torch.cuda.is_available() else "cpu")
print("pixel_high", PIXEL_HIGH)

sampel 809
device cuda
pixel_high 235


## Re-extract Handcrafted Features (Threshold Baru)

Fitur handcrafted diekstrak ulang dengan pixel_high=235. Pada palm, threshold 235 membuang pixel near-glare yang sebelumnya lolos di threshold 240, tetapi tidak terlalu agresif seperti 220 yang membuang 37.7% pixel valid.

In [2]:
handcrafted = features.extract_handcrafted_features(manifest, pixel_high=PIXEL_HIGH)
handcrafted.to_csv(output_dir / "handcrafted_features_t235.csv", index=False)
print("handcrafted features", handcrafted.shape)

handcrafted features (809, 28)


## Extract Deep Embeddings per Backbone

In [3]:
backbone_resnet = features.EmbeddingBackbone(backbone_name="resnet18", seed=42)
embeddings_resnet, uids_resnet = features.extract_deep_embeddings(
    manifest, model=backbone_resnet, size=backbone_resnet.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_resnet18_t235.npy", embeddings_resnet)
pd.DataFrame({"uid": uids_resnet}).to_csv(output_dir / "deep_embeddings_resnet18_t235_uids.csv", index=False)
print("resnet18", embeddings_resnet.shape)

resnet18 (809, 256)


In [4]:
backbone_dinov2 = features.EmbeddingBackbone(backbone_name="dinov2_small", seed=42)
embeddings_dinov2, uids_dinov2 = features.extract_deep_embeddings(
    manifest, model=backbone_dinov2, size=backbone_dinov2.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_dinov2s_t235.npy", embeddings_dinov2)
pd.DataFrame({"uid": uids_dinov2}).to_csv(output_dir / "deep_embeddings_dinov2s_t235_uids.csv", index=False)
print("dinov2_small", embeddings_dinov2.shape)

dinov2_small (809, 256)


In [5]:
backbone_siglip = features.EmbeddingBackbone(backbone_name="siglip_base_patch16_224", seed=42)
embeddings_siglip, uids_siglip = features.extract_deep_embeddings(
    manifest, model=backbone_siglip, size=backbone_siglip.input_size, pixel_high=PIXEL_HIGH,
)
np.save(output_dir / "deep_embeddings_siglip_t235.npy", embeddings_siglip)
pd.DataFrame({"uid": uids_siglip}).to_csv(output_dir / "deep_embeddings_siglip_t235_uids.csv", index=False)
print("siglip_base", embeddings_siglip.shape)

siglip_base (809, 256)


## Backbone Comparison: Path B (Deep Only) and Full Fusion

In [6]:
def _severity_kappa(oof):
    severity_valid = oof["severity_true"] >= 0
    if not severity_valid.any():
        return float("nan")
    return cohen_kappa_score(oof.loc[severity_valid, "severity_true"], oof.loc[severity_valid, "severity_pred"])


backbone_configs = {
    "resnet18": (embeddings_resnet, uids_resnet),
    "dinov2_small": (embeddings_dinov2, uids_dinov2),
    "siglip_base": (embeddings_siglip, uids_siglip),
}

summary_rows = []
all_results = {}

for config_type, use_hc in [("path_b_deep", False), ("full_fusion", True)]:
    print(f"\n--- {config_type} ---")
    for backbone_name, (embeddings, uids) in backbone_configs.items():
        result = train.run_kfold(
            manifest, handcrafted, embeddings, uids,
            n_splits=5, epochs=60, use_handcrafted=use_hc, use_deep=True,
            weight_severity_classes=True,
        )
        key = f"{backbone_name}_{config_type}"
        all_results[key] = result
        fm = result["fold_metrics"]
        summary_rows.append({
            "backbone": backbone_name,
            "config": config_type,
            "mae": fm["mae"].mean(),
            "accuracy": fm["accuracy"].mean(),
            "auc": fm["auc"].mean(),
            "f1": fm["f1"].mean(),
            "balanced_accuracy": fm["balanced_accuracy"].mean(),
            "severity_kappa": _severity_kappa(result["oof"]),
        })
        print(
            f"  {backbone_name:20s}",
            f"MAE {fm['mae'].mean():.3f}",
            f"acc {fm['accuracy'].mean():.3f}",
            f"AUC {fm['auc'].mean():.3f}",
            f"F1 {fm['f1'].mean():.3f}",
        )


--- path_b_deep ---


  resnet18             MAE 0.751 acc 0.640 AUC 0.732 F1 0.608


  dinov2_small         MAE 0.793 acc 0.644 AUC 0.694 F1 0.593


  siglip_base          MAE 0.740 acc 0.645 AUC 0.734 F1 0.618

--- full_fusion ---


  resnet18             MAE 0.760 acc 0.645 AUC 0.736 F1 0.594


  dinov2_small         MAE 0.835 acc 0.649 AUC 0.702 F1 0.556


  siglip_base          MAE 0.744 acc 0.650 AUC 0.730 F1 0.611


## Consolidated Comparison

In [7]:
summary_table = pd.DataFrame(summary_rows)
summary_table.to_csv(output_dir / "backbone_experiment_comparison.csv", index=False)
print(summary_table.sort_values("mae").round(4).to_string(index=False))

    backbone      config    mae  accuracy    auc     f1  balanced_accuracy  severity_kappa
 siglip_base path_b_deep 0.7404    0.6452 0.7338 0.6179             0.7054          0.2125
 siglip_base full_fusion 0.7441    0.6501 0.7303 0.6110             0.7001          0.2386
    resnet18 path_b_deep 0.7506    0.6403 0.7319 0.6078             0.6958          0.1914
    resnet18 full_fusion 0.7602    0.6452 0.7363 0.5939             0.6847          0.2120
dinov2_small path_b_deep 0.7925    0.6440 0.6941 0.5930             0.6839          0.2098
dinov2_small full_fusion 0.8348    0.6490 0.7024 0.5563             0.6566          0.2271


## Hyperparameter Search on Best Backbone

Backbone dengan MAE terendah di tabel di atas dioptimasi lewat Optuna. Untuk palm, MAE menjadi objective utama karena distribusi kelas lebih seimbang dibanding nail.

In [8]:
import optuna

best_backbone_row = summary_table.sort_values("mae").iloc[0]
best_backbone_name = best_backbone_row["backbone"]
best_config_type = best_backbone_row["config"]
best_embeddings, best_uids = backbone_configs[best_backbone_name]
use_handcrafted_best = best_config_type == "full_fusion"

print(f"optimasi backbone: {best_backbone_name}, config: {best_config_type}")


def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 30, 120)
    weight_classification = trial.suggest_float("weight_classification", 0.5, 3.0)
    weight_severity = trial.suggest_float("weight_severity", 0.1, 1.0)
    focal_gamma = trial.suggest_float("focal_gamma", 0.5, 3.0)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    trunk_dim = trial.suggest_categorical("trunk_dim", [64, 128, 256])
    attention_dim = trial.suggest_categorical("attention_dim", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    result = train.run_kfold(
        manifest, handcrafted, best_embeddings, best_uids,
        n_splits=5, epochs=epochs, batch_size=batch_size, learning_rate=learning_rate,
        loss_weights=(1.0, weight_classification, weight_severity),
        trunk_dim=trunk_dim, attention_dim=attention_dim, dropout=dropout, focal_gamma=focal_gamma,
        use_handcrafted=use_handcrafted_best, use_deep=True, weight_severity_classes=True,
    )
    mae_mean = float(result["fold_metrics"]["mae"].mean())
    trial.set_user_attr("mae_mean", mae_mean)
    trial.set_user_attr("auc_mean", float(result["fold_metrics"]["auc"].mean()))
    trial.set_user_attr("accuracy_mean", float(result["fold_metrics"]["accuracy"].mean()))
    return mae_mean


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=40)

print("best MAE", round(study.best_value, 4))
print("best AUC", round(study.best_trial.user_attrs["auc_mean"], 4))
print("best accuracy", round(study.best_trial.user_attrs["accuracy_mean"], 4))
print("best params", study.best_params)

[I 2026-09-30 14:00:22,909] A new study created in memory with name: no-name-7df30959-c12f-4ba6-9139-29b2b1dc2bb4


optimasi backbone: siglip_base, config: path_b_deep


[I 2026-09-30 14:00:34,395] Trial 0 finished with value: 0.7490412831306458 and parameters: {'learning_rate': 0.0007216029177171684, 'epochs': 109, 'weight_classification': 1.7022388228435001, 'weight_severity': 0.8990901971588595, 'focal_gamma': 0.8004156046064497, 'dropout': 0.4816998600324701, 'trunk_dim': 128, 'attention_dim': 128, 'batch_size': 64}. Best is trial 0 with value: 0.7490412831306458.


[I 2026-09-30 14:00:47,714] Trial 1 finished with value: 0.7457014441490173 and parameters: {'learning_rate': 0.0002059539877531595, 'epochs': 116, 'weight_classification': 0.5383791143470524, 'weight_severity': 0.3960187787783157, 'focal_gamma': 2.4697467984000934, 'dropout': 0.46236203762842965, 'trunk_dim': 256, 'attention_dim': 32, 'batch_size': 64}. Best is trial 1 with value: 0.7457014441490173.


[I 2026-09-30 14:00:52,326] Trial 2 finished with value: 0.7591947913169861 and parameters: {'learning_rate': 0.001675059314473806, 'epochs': 74, 'weight_classification': 2.3970070418845317, 'weight_severity': 0.5710518930675373, 'focal_gamma': 0.690892972440353, 'dropout': 0.17054722466337058, 'trunk_dim': 256, 'attention_dim': 128, 'batch_size': 128}. Best is trial 1 with value: 0.7457014441490173.


...

[I 2026-09-30 14:05:04,946] Trial 36 finished with value: 0.7478233456611634 and parameters: {'learning_rate': 0.0016512189742942881, 'epochs': 108, 'weight_classification': 1.5480197294167124, 'weight_severity': 0.47208434546319755, 'focal_gamma': 1.8662015050701857, 'dropout': 0.2581035772029014, 'trunk_dim': 64, 'attention_dim': 128, 'batch_size': 128}. Best is trial 20 with value: 0.7356521964073182.


[I 2026-09-30 14:05:10,778] Trial 37 finished with value: 0.7427166223526 and parameters: {'learning_rate': 0.0040649027252511695, 'epochs': 97, 'weight_classification': 2.9972024875373355, 'weight_severity': 0.7962099193394266, 'focal_gamma': 1.480785231468874, 'dropout': 0.4594377862890353, 'trunk_dim': 128, 'attention_dim': 128, 'batch_size': 128}. Best is trial 20 with value: 0.7356521964073182.


[I 2026-09-30 14:05:12,911] Trial 38 finished with value: 0.7836509823799134 and parameters: {'learning_rate': 0.0010326139569433134, 'epochs': 34, 'weight_classification': 1.8111635832600697, 'weight_severity': 0.9962667049641348, 'focal_gamma': 2.561338920034928, 'dropout': 0.2162455440248598, 'trunk_dim': 128, 'attention_dim': 64, 'batch_size': 128}. Best is trial 20 with value: 0.7356521964073182.


[I 2026-09-30 14:05:17,427] Trial 39 finished with value: 0.7411512732505798 and parameters: {'learning_rate': 0.0028039683947102823, 'epochs': 71, 'weight_classification': 2.082408943757232, 'weight_severity': 0.6646654257427569, 'focal_gamma': 0.9867496422857253, 'dropout': 0.3005122168329399, 'trunk_dim': 128, 'attention_dim': 128, 'batch_size': 128}. Best is trial 20 with value: 0.7356521964073182.


best MAE 0.7357
best AUC 0.7316
best accuracy 0.6464
best params {'learning_rate': 0.0026540938474324555, 'epochs': 86, 'weight_classification': 1.7066085001816196, 'weight_severity': 0.9792950091303262, 'focal_gamma': 1.6729103221472919, 'dropout': 0.30912818204531783, 'trunk_dim': 128, 'attention_dim': 128, 'batch_size': 128}


## Retrain with Best Hyperparameters

In [9]:
best_params = study.best_params
tuned_result = train.run_kfold(
    manifest, handcrafted, best_embeddings, best_uids,
    n_splits=5,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    learning_rate=best_params["learning_rate"],
    loss_weights=(1.0, best_params["weight_classification"], best_params["weight_severity"]),
    trunk_dim=best_params["trunk_dim"],
    attention_dim=best_params["attention_dim"],
    dropout=best_params["dropout"],
    focal_gamma=best_params["focal_gamma"],
    use_handcrafted=use_handcrafted_best, use_deep=True, weight_severity_classes=True,
)

tuned_fm = tuned_result["fold_metrics"]
print("tuned MAE", round(tuned_fm["mae"].mean(), 4))
print("tuned accuracy", round(tuned_fm["accuracy"].mean(), 4))
print("tuned AUC", round(tuned_fm["auc"].mean(), 4))
print("tuned F1", round(tuned_fm["f1"].mean(), 4))
print("tuned severity_kappa", round(_severity_kappa(tuned_result["oof"]), 4))

tuned MAE 0.7473
tuned accuracy 0.6489
tuned AUC 0.73
tuned F1 0.6223
tuned severity_kappa 0.2154


## Save Checkpoints and Results

In [10]:
tuned_result["oof"].to_csv(output_dir / f"multitask_oof_{best_backbone_name}_{best_config_type}_tuned.csv", index=False)

for fold_index, fold_model in enumerate(tuned_result["models"]):
    checkpoint_path = artifact_dir / f"multitask_{best_backbone_name}_{best_config_type}_tuned_fold{fold_index}.pt"
    torch.save(fold_model.state_dict(), checkpoint_path)

with open(output_dir / "backbone_experiment_best.json", "w") as handle:
    json.dump({
        "backbone": best_backbone_name,
        "config": best_config_type,
        "use_handcrafted": use_handcrafted_best,
        "pixel_high": PIXEL_HIGH,
        "best_params": best_params,
        "mae": study.best_value,
        "auc": study.best_trial.user_attrs["auc_mean"],
        "accuracy": study.best_trial.user_attrs["accuracy_mean"],
    }, handle, indent=2)

print(f"checkpoint disimpan untuk {best_backbone_name} {best_config_type}")

checkpoint disimpan untuk siglip_base path_b_deep
